# GeoGebra applet interaction from Python with errors

In [1]:
# this magic for develop only
%load_ext autoreload
%autoreload 2

In [2]:
from ggblab import GeoGebra

In [4]:
from ggblab.errors import GeoGebraError, GeoGebraSyntaxError, GeoGebraSemanticsError, GeoGebraAppletError

In [19]:
ggb = GeoGebra()

ggb.check_syntax = True
ggb.check_semantics = True

Using local cached file: xsd/common.xsd


In [53]:
from ggblab.utils import flatten

## GeoGebra command tokenizer as syntax checker

In [20]:
# tokenizer delimitate a command string with spaces, parentheses, and brackets
r = ggb.parser.tokenize_with_commas("Circle(Midpoint(A, B), 1)")
r

['Circle', ['Midpoint', ['A', ',', 'B'], ',', '1']]

In [21]:
# reconstruct command string from tokens
ggb.parser.reconstruct_from_tokens(r)

'Circle(Midpoint(A, B), 1)'

In [22]:
try:
    r = ggb.parser.tokenize_with_commas("", extract_commands=True)
except Exception as e:
    print(f"{type(e).__name__}: {e}")
else:
    print(r)

{'tokens': [], 'commands': set()}


In [23]:
try:
    r = ggb.parser.tokenize_with_commas("Circle(A, ", extract_commands=True)
except Exception as e:
    print(f"{type(e).__name__}: {e}")
else:
    print(r)

ValueError: Mismatched parentheses/brackets in input string.


In [29]:
try:
    r = ggb.parser.tokenize_with_commas("Circle(A, ))", extract_commands=True)
except Exception as e:
    print(f"{type(e).__name__}: {e}")
else:
    print(r)

ValueError: Mismatched parentheses/brackets in input string.


In [27]:
# open GeoGebra Widget on left-side
await ggb.init()

In [77]:
# Pre-flight validation errors (caught before execution)
try:
    await ggb.command("Circle(A, ")
except GeoGebraSyntaxError as e:
    print(f"Syntax error: {e}")
except GeoGebraSemanticsError as e:
    print(f"Missing objects: {e.missing_objects}")

Syntax error: Syntax error in command 'Circle(A, ': Mismatched parentheses/brackets in input string.


## GeoGebra command semantics checker

In [25]:
# command_cache store command names to distinguish from label names
ggb.parser.command_cache.get_all()

{'Angle': 48,
 'Area': 6,
 'Circle': 74,
 'Distance': 30,
 'Intersect': 160,
 'Line': 149,
 'Midpoint': 26,
 'PerpendicularLine': 128,
 'Point': 52,
 'Polygon': 318,
 'Ray': 104,
 'Segment': 574,
 'Translate': 18,
 'Vector': 87,
 'cos': 12,
 'sin': 2,
 'sin²': 10}

In [58]:
ggb.parser.command_cache.increment('Circle')

In [59]:
'Circle' in ggb.parser.command_cache

True

In [74]:
ggb.parser.command_cache.clear()
ggb.parser.command_cache.increment(['Angle', 'Area', 'Circle', 'Distance', 'Intersect', 'Line', 'Midpoint', 
                                    'PerpendicularLine', 'Point', 'Polygon', 'Ray', 'Segment', 'Translate', 'Vector'])
ggb.parser.command_cache.get_all()

{'Angle': 1,
 'Area': 1,
 'Circle': 1,
 'Distance': 1,
 'Intersect': 1,
 'Line': 1,
 'Midpoint': 1,
 'PerpendicularLine': 1,
 'Point': 1,
 'Polygon': 1,
 'Ray': 1,
 'Segment': 1,
 'Translate': 1,
 'Vector': 1}

In [31]:
# Pre-flight validation errors (caught before execution)
try:
    await ggb.command("Circle(A, B)")
except GeoGebraSyntaxError as e:
    print(f"Syntax error: {e}")
except GeoGebraSemanticsError as e:
    print(f"Missing objects: {e.missing_objects}")

Missing objects: ['A', 'B']


In [76]:
# Runtime errors from GeoGebra applet (caught after execution)
try:
    await ggb.command("Circle()")
except GeoGebraSemanticsError as e:
    print(f"{e}")
except GeoGebraAppletError as e:
    print(f"Applet error: {e}")

Applet error: GeoGebra applet error: Command Circle:
Illegal number of arguments: 0

Syntax:
Circle( <Point>, <Radius Number> )
Circle( <Point>, <Segment> )
Circle( <Point>, <Point> )
Circle( <Point>, <Point>, <Point> )
Circle( <Line>, <Point> )
Circle( <Point>, <Radius>, <Direction> )
Circle( <Point>, <Point>, <Direction> ) [AppletError]
